In [ ]:
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_COLAB:
    print("Colab environment. Make sure the HW_DIR below is matches what you used in the startup notebook.")
    print("If you are asked to restart the runtime, you do not need to do so.")
    print("If you decide to restart the runtime, you will need to re-run this cell.")
    from google.colab import drive
    drive.mount("/content/drive")
    HW_DIR = "/content/drive/MyDrive/CS189/hw/hw2"
    os.chdir(HW_DIR)
    %pip install -r colab_requirements.txt
elif IS_DATABRICKS:
    print("Databricks environment. Make sure your base environment is ML v5.")
    %pip install -r databricks_requirements.txt
else:
    print("Local development. Make sure your kernel has everything in requirements.txt.")


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("chatbot_arena.ipynb")

<h1>Homework 2: Welcome to the Arena</h1>

In this homework you will get more experience with logistic regression in two very different settings: creating leaderboards and predicting model responses.

We will be taking real data from [Arena](https://arena.ai/), a popular platform for crowsourcing evaluations of large language models and recreating their leaderboards, with a few fun extra steps along the way. The chats can be viewed interactively by accessing [ChatBot-Arena-Viewer](https://huggingface.co/spaces/BerkeleyML/Chatbot-Arena-Viewer) through Hugging Face. Arena spun out of a research project at Berkeley led by Professor Gonzalez, became a standard method for evaluating LLMs, and is now a company! While there is still a lot of public debate over using [Arena AI's data as LLM benchmarks](https://techcrunch.com/2024/09/05/the-ai-industry-is-obsessed-with-chatbot-arena-but-it-might-not-be-the-best-benchmark/) — and a lot of public debate over AI benchmarks in general! — Arena AI is still widely used to compare new models. So don't let anyone tell you logistic regression isn't valuable, it's worth at least [$1.7 billion](https://techcrunch.com/2026/01/06/lmarena-lands-1-7b-valuation-four-months-after-launching-its-product/).

---

## Due Date: Friday, October 9, 11:59 PM

This assignment is due on **Friday, October 9, 11:59 PM**. You must submit your work to the programming assignment by this deadline. Please refer to the syllabus for the [Slip Day policy](https://eecs189.org/fa26/syllabus/#slip-days). No late submissions will be accepted beyond the details outlined in the Slip Day policy.

### Submission Tips:
- **Plan ahead**: We strongly encourage you to submit your work several hours before the deadline. This will give you ample time to address any submission issues.
- **Reach out for help early**: If you encounter difficulties, contact course staff well before the deadline. While we are happy to assist with submission issues, we cannot guarantee responses to last-minute requests.

---

### Key Learning Objectives:
1. Learn how to evaluate large language models (LLMs) using pairwise comparison data from LMArena (now known as Arena AI)
2. Analyze battle distributions and compute win rates
3. Use latent semantic analysis and Gaussian mixture models to discover prompt topics
4. Apply the Bradley–Terry model to build leaderboards
5. Train logistic regression models to predict LLM win probabilities
6. Practice analyzing conversational data, extracting stylistic features, and exploring how stylistic choicces can confound LLM evaluation (style vs. content)
7. Build custom features (length, punctuation, phrase presence, etc.) and integrate them into ranking models  

---

<div align="center">

### Grading Breakdown

| Question | Manual Grading? | Points |
|----------|-----------------|--------|
| 1a       | No              | 2      |
| 1b       | No              | 1      |
| 1c       | Yes             | 2      |
| 2a       | No              | 2      |
| 2b       | Yes             | 2      |
| 3a       | Yes             | 2      |
| 3b       | No              | 4      |
| 3c       | No              | 1      |
| 3d       | No              | 2      |
| 3e       | Yes             | 2      |
| 4a       | No              | 2      |
| 4b       | No              | 2      |
| 4c       | No              | 3      |
| 4d       | Yes             | 1      |
| 5a       | No              | 2      |
| 5b       | No              | 2      |
| 6a       | No              | 2      |
| 6b       | No              | 2      |
| 6c       | No              | 3      |
| 7a       | No              | 4      |
| 7b       | Yes             | 4      |
| **Total**|                 | **47** |

</div>

---

### Instructions:
1. Carefully read each question and its requirements.
2. Complete all TODOs in the notebook. You may add extra lines of code if needed.
3. For manual questions, write clear and concise responses in the provided `q*_answer` strings, then run those cells to save the responses under `written/`.
4. Test your code thoroughly to ensure it meets the requirements.


---

### Autograding
- After each coding question, run `grader.check("q...")` to execute the **public** tests.
- Keep the `tests/` folder next to this notebook. It ships in the homework repo. If staff announces a test update, use the cell below to restore only `tests/` from the announced tag.
- The autograder also runs additional tests that are not in this handout. You will see pass/fail and error messages for those tests when you submit.
- Submit your coding `.ipynb` and output `.png` and `.txt` files to the **programming** assignment. The export cell at the end of the notebook packages everything into a `.zip` you can submit and warns you of missing files.
- Both the final export cell and the autograder expect the assignment notebook to have the same name as the input to `otter.Notebook(...)` in the Otter initialization cell at the top of the notebook. Do not rename your notebook, otherwise the final export and the autograder may run into errors processing your submission!
- A PDF of your manually graded responses is uploaded to the **associated paper assignment**. You do not need to select pages.

Good luck!


## Updating public tests

If staff announces a public-test update, set `TESTS_VERSION` to the announced tag, set `REPULL_TESTS = True` in the cell below, run it, then set it back to `False`. This restores only `tests/` from the tag and leaves your notebook and other files unchanged.

In [ ]:
# PUBLIC TEST VERSION
TESTS_VERSION = "hw2-public-v1"
REPULL_TESTS = False

if REPULL_TESTS:
    !git fetch origin tag {TESTS_VERSION}
    !git restore --source={TESTS_VERSION} -- tests/


### **IMPORTANT:**
- Do not change the random seed values.
- Save the notebook before exporting.
- You do not have to save or submit any trained models for this homework assignment.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.mixture import GaussianMixture
import time

pd.options.plotting.backend = "plotly"

# Set random seeds for reproducible results
SEED = 189
np.random.seed(SEED)
IS_GRADING_ENV = os.getenv("IS_GRADING_ENV") == "true"

WRITTEN_DIR = "written"
os.makedirs(WRITTEN_DIR, exist_ok=True)

with open(os.path.join("tests", "written_files.json"), encoding="utf-8") as f:
    EXPECTED_WRITTEN_FILES = json.load(f)

def save_figure(fig, question_name):
    """Save a Plotly or Matplotlib figure under written/."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.png")
    try:
        if hasattr(fig, "write_image"):
            fig.write_image(path)
        else:
            fig.savefig(path, bbox_inches="tight")
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path

def save_written_answer(answer, question_name):
    """Save a written response under written/."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.txt")
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(str(answer).strip() + "\n")
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path

def warn_missing_written_files():
    """Report expected written artifacts that are missing locally."""
    missing = [name for name in EXPECTED_WRITTEN_FILES if not os.path.isfile(os.path.join(WRITTEN_DIR, name))]
    if missing:
        print("WARNING: Missing expected files in written/:")
        for name in missing:
            print(f"  - {name}")
        print("Re-run the cells that save these artifacts before exporting.")
    else:
        print(f"All {len(EXPECTED_WRITTEN_FILES)} expected files found in written/.")
    return missing


<div style="text-align: center;">
  <img src="https://i.imgur.com/EAYpMiZ.png" alt="Arena AI" style="display: block; margin-left: auto; margin-right: auto; width: 60%;">
</div>


# What is Arena AI?

Arena AI (previously known as LMArena or Chatbot Arena) is a platform that evaluates generative large language models (e.g. chatbots or image generation models) through anonymous, crowd-sourced pairwise comparisons. Users enter prompts for two anonymous LLMs to respond to and vote on the LLMs that gave the better response, in which the LLM's identities are revealed (shown below). Users can also choose LLMs to test themselves, but for the purposes of this homework we will only focus on the anonymous side-by-side comparisons, which we call **"battles"** -  since those are what are used to calcuate the leaderboard.

<div style="text-align: center;">
  <img src="https://i.imgur.com/sVXKlUP.png" alt="An example of an Arena AI battle for the question 'what's the best computer science course at Berkeley?'" style="display: block; margin-left: auto; margin-right: auto; width: 100%; border-radius: 1%;">
</div>


In this homework we will investigate what these battles look like, how we can use these pairwise comparisons to get a leaderboard, and how we can find certain features of LLM responses that have an influence on preference.

Although it is not required for this notebook, the [Chatbot Arena paper](https://arxiv.org/abs/2403.04132) can provide good intuition on how to answer the free response questions. 

# Question 1: Data Prep and Exploration

First, let's load a set of publicly released arena battles from [Hugging Face](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-100k) — a popular website for sharing machine learning datasets and models. They also have a ton of great visualization tools with [Gradio](https://www.gradio.app/docs), which you will also get some experience with in this homework.


**Note**: Before you get started with the homework, we recommend you make a [Hugging Face account](https://huggingface.co/welcome) to play around with creating or modifying our data visualization apps! It is also a great general hub for downloading the majority of popular datasets in machine learning. Once you make the account, generate a [user access token](https://huggingface.co/docs/hub/security-tokens). This should be similar to how you generate a token for your Git account.

To log into Hugging Face via the browser, uncomment the cells below and run it. You should be taken to Hugging Face's wbesite to log in!

In [ ]:
# Optional: if Hugging Face asks you to authenticate, uncomment and run these lines.
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset (this will take a few minutes to download)
# If you haven't logged in via your HF account, it may throw a warning to create a HF_TOKEN for faster downloads;
# however, this shouldn't block you from accessing the dataset.
ds = load_dataset("lmarena-ai/arena-human-preference-100k")
battles = ds['train'].to_pandas()
battles = battles[battles["dedup_tag"].apply(lambda x: x.get("sampled", False))]

Now let's look at the format of this data.

Printing out the first row we see there are many fields with the most important being:

<ul>
  <li><strong>question_id (str):</strong> the ID of that battle</li>
  <li><strong>model_a, model_b (str):</strong> the LLMs which participated in this battle</li>
  <li><strong>winner (str):</strong> which response the user preferred. Can be <code>model_a</code>, <code>model_b</code>, <code>tie</code>, and <code>tie (both bad)</code>. Notice how the data has 2 types of ties: just tie, and tie (both bad).</li>
  <li><strong>conversation_a (dict):</strong> the conversation between the user and model a</li>
  <li><strong>conversation_b (dict):</strong> the conversation between the user and model b (note that the user turns in conversation_a and conversation_b are the same since this is a side by side comparison)</li>
  <li><strong>turn (int):</strong> number of turns in the conversation (1 turn means the user asked 1 question, 2 turns means the user asked a question, got an answer, then asked another question, got the response, then voted)</li>
  <li><strong>language (str):</strong> the language of the user prompt</li>
</ul>

We also have some other columns that may be useful for us later:

<ul>
  <li><strong>is_code (bool):</strong> either the prompt, the response, or both contains code</li>
  <li><strong>is_refusal (bool):</strong> one of the models refused to answer (usually this is because the model thinks it would be unethical to answer)</li>
  <li><strong>dedup_tag (dict):</strong> indicates whether the prompt appears very often (high_frequency) and if it does, whether it will be sampled (subsampled). We subsample these high frequency prompts so that common questions don't overly influence the leaderboard.</li>
  <li><strong>category_tag (dict):</strong> tags for question type (e.g. math and instruction following). These are assigned via an LLM labeler, more details on what the categories are in this <a href="https://blog.lmarena.ai/blog/2024/arena-category/">blog post</a> (the criteria tags correspond to the hard prompts category described in the blog post)</li>
</ul>

In [ ]:
print("Columns:\n*", "\n* ".join(battles.columns))
print("=" * 20)
example = battles.iloc[4]
print(f"Conversation A (model = {example['model_a']}):")
print(example['conversation_a'])
print("=" * 20)
print(f"Conversation B (model = {example['model_b']}):")
print(example['conversation_b'])
print("=" * 20)
print("Category Tag:")
print(example['category_tag'])

The cell below produces a bar plot of how many battles each model was invovled in. Notice that certain LLMs participate in more battles. This is due to two reasons:
1. Several different matching and sampling algorithms were used. Arena AI/LMArena employs weighted sampling methods, which assign greater weights to better models.
2. Since models are added to the arena when they come out, some models have been on the arena for many months while others have only been on for a few weeks.

In [ ]:
battle_count_barplot = pd.concat([battles["model_a"], battles["model_b"]]).value_counts().plot.bar(title="Battle Count for Each Model", text_auto=True)
battle_count_barplot.update_layout(xaxis_title="Model", yaxis_title="Battle Count", height=400, showlegend=False)
if not IS_GRADING_ENV:
    battle_count_barplot.show()

## Question 1a: Select the Most-Frequently Battled Models

Let's just focus on the top 20 LLMs by battle count for now. Build a list of the top 20 models by battle count, making sure to count both when a model was `model_a` or `model_b`. Sort the list by battle count in descending order.

In [ ]:
# TODO
selected_lmarena_models = ...

In [ ]:
grader.check("q1a")

## Question 1b: Filtering Battles to Top Models

Now, filter the data to only select battles where both `model_a` and `model_b` were one of the top 20 models we got from question 1a. Fill in the `subselect_battles` function to return a DataFrame:
- the dataframe should only contain battles between the selected models
- the dataframe should also filter out battles that resulted in either tie type. We want to train logistic regression models to predict LLM strengths later on, and we will need battles that have clear winners and losers as the training data.

**Hint:** You might find it helpful to use a boolean array while filtering your dataframes.

In [ ]:
def subselect_battles(
    battles: pd.DataFrame, 
    selected_models: list[str]
) -> pd.DataFrame:
    """
    Filters the battles DataFrame to only include battles between the selected models.
    Returns a dataframe filtered by models with ties removed.
    """
    ...
    return selected_battles_no_ties
  
selected_battles_no_ties = subselect_battles(battles, selected_lmarena_models)

In [ ]:
grader.check("q1b")

We’re going to visualize how often each pair of LLMs battled each other, similarly to the heatmap on the right side of **Figure 2** in the [LMArena paper](https://arxiv.org/pdf/2403.04132). We’ll sort models by their total number of battles and include the top **N** (default 20) models in our heatmap. Each cell of the heatmap shows the battle counts for the *(Model A, Model B)* match-up.

In [ ]:
def visualize_battle_count(battles, title, show_num_models=20):
    """
    Staff-provided helper!
    
    Input:
        battles : pd.DataFrame with columns ['model_a','model_b', ...]
        title   : str, title for the plot
        show_num_models : int, how many top models (by total battle count) to display

    Output:
        battle_count_heatmap_fig : plotly.graph_objects.Figure heatmap of symmetric battle counts
    """
    ptbl = pd.pivot_table(battles, index="model_a", columns="model_b", aggfunc="size", fill_value=0)
    battle_counts = ptbl + ptbl.T
    ordering = battle_counts.sum().sort_values(ascending=False).index
    ordering = ordering[:show_num_models]

    battle_count_heatmap_fig = px.imshow(
        battle_counts.loc[ordering, ordering],
        title=title,
        text_auto=True
    )
    battle_count_heatmap_fig.update_layout(
        xaxis_title="Model B",
        yaxis_title="Model A",
        xaxis_side="top",
        height=1000,
        width=1000,
        title_y=0.07,
        title_x=0.5,
        font=dict(size=10)
    )
    battle_count_heatmap_fig.update_traces(
        hovertemplate="Model A: %{y}<br>Model B: %{x}<br>Count: %{z}<extra></extra>"
    )
    return battle_count_heatmap_fig

In [ ]:
# Visualize selected battles with ties filtered out
if not IS_GRADING_ENV:
    battle_count_heatmap_fig = visualize_battle_count(selected_battles_no_ties, "Battle Count for Each Combination of Models (without Ties)")
    battle_count_heatmap_fig.show()

## **Question 1c: Explain Arena's Model-Pairing Strategy**
We see many battles between top models (e.g., Claude, GPT, Gemini), while smaller models (e.g., Llama-3-8B) have fewer battles. This is because LMArena employs weighted sampling methods, which assign greater weights to better models and results in better models being picked more.

**Question:** Why might LMArena pair strong models vs. strong models more often than strong vs. weaker/smaller models? Think about statistical power, how easily you can compare different models, and ranking uncertainty.

Write your response in `q1c_answer` and run the answer cell to save it for the written submission.

<!-- BEGIN QUESTION -->



In [ ]:
# TODO: Write your answer in `q1c_answer`, then run this cell to save it.
q1c_answer = """
YOUR ANSWER HERE
"""
save_written_answer(q1c_answer, "q1c")

<!-- END QUESTION -->

# Question 2: Model Rankings

Now that we have explored our data, let's consider how to use these pairwise battles to rank the models by preference. Our goal is to assign a "strength" parameter to each LLM that quantifies how likely it is to win against others.

Given we analyzed $M=20$ models and $N=40k$ battles ($26k$ excluding ties), we want to estimate a skill parameter $S_i$ for each model $i \in \{1, \ldots, M\}$. This parameter $S_i$ should reflect the overall ability of model $i$ to be preferred over other models.

Before we move on to more sophisticated probabilistic models that estimate these strength parameters, let’s build some intuition by starting with a simpler metric: the average win rate.

Average win rate is simple: an model's average win rate is the proportion of battles they competed in which resulted in them winning. 

## Question 2a: Compute Pairwise Win Fractions

For a pair of models, the pairwise win fraction is the fraction of their head-to-head battles won by the first model. Later, we will compute each model's overall win rate by averaging its pairwise win fractions across opponents.

Implement the `compute_pairwise_win_fraction` function, which:
1. Takes in a dataframe of battles and derive the model list from the `model_a` and `model_b` columns of that input; do not reference global variables such as `battles`, `selected_battles_no_ties`, or `selected_lmarena_models` inside the function. Later in the notebook, we will pass different dataframes of battles to this function, so this function should be able to flexibly compute pairwise win fractions from any passed-in `battles_df`.

2. For every ordered pair of models, calculates the fraction of their head-to-head battles won by the first model.

3. Returns a square DataFrame called `row_beats_col` where entry (i, j) is the fraction of times model i beats model j. The DataFrame's values must have a numeric floating-point dtype; construct the underlying arrays with `dtype=float` before converting them to a DataFrame. Any model pairings which do not have any battles should be given a `NaN` value. For instance, the diagonal of `row_beats_col` should be `NaN` because no model battles itself.

4. The rows and columns of your `row_beats_col` dataframe should be ordered by their average win rate against all other models (i.e. order from strongest to weakest models)

**Hints:**
- Build a mapping from each model name to a row and column index, then use NumPy arrays to count wins and total battles for every pairing. Convert the resulting win-fraction matrix to a DataFrame only at the end.
- Treat each battle as contributing to both ordered orientations: increment the total for `(model_a, model_b)` and for `(model_b, model_a)`. If `winner == "model_a"`, increment the win count for `(model_a, model_b)`; if `winner == "model_b"`, increment the win count for `(model_b, model_a)`. A tie contributes to the totals but to neither win count (the provided input for this question has ties removed).
- The returned DataFrame must contain numeric floating-point values. For instance, pass `dtype=float` when creating any arrays or cast the numeric values in your arrays/Dataframe before returning the result from this function.

In [ ]:
def compute_pairwise_win_fraction(battles_df):
    # TODO

    models = ...
    model_to_index = ...
    wins = ...
    totals = ...

    for battle in ...:

    win_fractions = ...
    row_beats_col_freq = ...
    model_names = ...
    row_beats_col = ...
    return row_beats_col

In [ ]:
grader.check("q2ai")

Below, we've defined a function to visualize how often **Model A** beats **Model B** in non-tied battles. The cell below uses the `compute_pairwise_win_fraction` you just implemented to create heatmap where each cell `(A, B)` displays the **fraction of A’s wins** over B.

Run the cell below (no code changes required!).

In [ ]:
# Run this cell, no code changes required :)
def visualize_pairwise_win_fraction(battles, title):
    """
    Staff-provided helper!

    Input:
        battles : pd.DataFrame of non-tied battles with ['model_a','model_b','winner', ...]
        title   : str
    Output:
        win_heatmap_fig : plotly Figure heatmap (cell (A,B) = fraction A beats B)
    """
    row_beats_col = compute_pairwise_win_fraction(battles)
    win_heatmap_fig = px.imshow(
        row_beats_col,
        color_continuous_scale='RdBu',
        text_auto=".2f",
        title=title
    )
    win_heatmap_fig.update_layout(
        xaxis_title=" Model B: Loser",
        yaxis_title="Model A: Winner",
        xaxis_side="top",
        height=900,
        width=900,
        title_y=0.07,
        title_x=0.5
    )
    win_heatmap_fig.update_traces(
        hovertemplate="Model A: %{y}<br>Model B: %{x}<br>Fraction of A Wins: %{z}<extra></extra>"
    )
    return win_heatmap_fig

if not IS_GRADING_ENV:
    win_heatmap_fig = visualize_pairwise_win_fraction(
        selected_battles_no_ties,
        title="Fraction of Model A Wins for All Non-tied A vs. B Battles"
    )
    win_heatmap_fig.show()

## Question 2b: Analyze Average Win Rates

Now we can just average the win rate of model $i$ against all other models to get an estimate of strength.

The cell below computes and visualizes the average win rate of each model against all others.

In the chart, we see that some models have very similar average win rates. For example, some GPT, Claude, and Llama variants sit close together. On the other hand, smaller models like llama-3-8b and gemma-2-9b fall noticeably behind. Parameter count alone does not determine model quality: architecture, training data, compute, and post-training all matter, and parameter counts for closed models are often not publicly available.

In [ ]:
def get_pairwise_win_rate(battles):
    """
    Staff-provided helper!

    Collapse the pairwise win-fraction matrix into a per-model leaderboard by
    averaging each model's row, skipping NaN match-ups (the diagonal and any
    unplayed pair) so every opponent counts equally regardless of battle count.

    Input:  battles : pd.DataFrame, one row per battle
    Output: pd.DataFrame ['model', 'win rate', 'rank'] sorted best to worst,
            rank 1 = best (tied models share a rank)
    """
    row_beats_col_freq = compute_pairwise_win_fraction(battles)
    pairwise_win_rate = row_beats_col_freq.mean(axis=1).reset_index()
    pairwise_win_rate.columns = ['model', 'win rate']

    # Rank (1 = best)
    pairwise_win_rate["rank"] = (
        pairwise_win_rate["win rate"].rank(ascending=False, method="dense").astype(int)
    )

    # Sort for plotting and freeze that order on the x-axis
    pairwise_win_rate = pairwise_win_rate.sort_values(by="win rate", ascending=False)

    return pairwise_win_rate

def get_pairwise_win_rate_plot(pairwise_win_rate_df, title=""):
    """
    Staff-provided helper!

    Draw the average win rates as a bar chart, one bar per model, labelled with
    the win rate to 2 decimals and showing the exact rate and rank on hover. The
    x-axis order is pinned to the input's row order so bars stay win-rate sorted.
    Input:  pairwise_win_rate_df : output of get_pairwise_win_rate
            title : str, chart title
    Output: plotly Figure (not displayed; call .show() on it)
    """
    model_order = pairwise_win_rate_df["model"].tolist()

    win_rate_fig = px.bar(
        pairwise_win_rate_df,
        x="model",
        y="win rate",
        title=title,
        text="win rate",
        hover_data=["rank"]
    )
    win_rate_fig.update_traces(texttemplate="%{text:.2f}", hovertemplate="<b>%{x}</b><br>win rate=%{y:.3f}<br>rank=%{customdata}")
    win_rate_fig.update_layout(
        yaxis_title="Average Win Rate",
        xaxis_title="Model",
        showlegend=False
    )
    win_rate_fig.update_xaxes(tickangle=45, categoryorder="array", categoryarray=model_order)
    return win_rate_fig

pairwise_win_rate = get_pairwise_win_rate(selected_battles_no_ties)
avg_win_barplot_fig = get_pairwise_win_rate_plot(pairwise_win_rate, title="Average Win Rate Against All Other Models (No Ties)")
if not IS_GRADING_ENV:
    avg_win_barplot_fig.show()

<!-- BEGIN QUESTION -->

**Question:** Identify two models with similar average win rates. Then explain one limitation of ranking models by this average.

**Hints:**
- Recall that the overall win rate averages across opponents, giving every opponent equal weight regardless of how many battles were played against them. Thus, a win fraction based on only a few battles counts just as much as one based on hundreds of battles.
- Models may not share the same set of opponents.

Write your response in `q2b_answer` and run the answer cell to save it for the written submission.

In [ ]:
# TODO: Write your answer in `q2b_answer`, then run this cell to save it.
q2b_answer = """
YOUR ANSWER HERE
"""
save_written_answer(q2b_answer, "q2b")

<!-- END QUESTION -->

# Question 3: Prompt Analysis

When evaluating models, it’s also often useful to understand the types of questions that are asked. By grouping similar prompts together, we can analyze which models perform well on certain categories and poorly on others. This helps uncover biases in leaderboards (e.g., a model may excel at coding questions but struggle with creative writing).

**First, let's identify the most frequent prompts.**

In the code cell below, we've already done the following:
* Extracted the first user message (prompt) from conversation_a
* Kept only the battles in English. Note that due to a caveat for LMArena data, some battles that were very short or only contained numbers did not have a labelled language (`battles['language'] == 'unknown'`). We also kept these unknown battles for our analysis.
* Filtered to only battles that used the selected LM arena models using your `subselect_battles` function from 1b.
* Printed the top 10 most frequent prompts for battles in English.

In [ ]:
# Staff-provided helper!
# Run this cell, no code changes required :)

def first_user_text(conv):
    return (conv[0].get("content") or "").strip()

battles['prompt'] = battles['conversation_a'].apply(first_user_text).fillna("")
eng_battles = battles[(battles['language'] == 'English') | (battles['language'] == 'unknown')]

eng_battles_no_ties = subselect_battles(eng_battles, selected_lmarena_models)

# Print the top 10 most common prompts along with their count and percentage of total prompts
top_prompts = eng_battles_no_ties["prompt"].value_counts().head(10)
for i, (prompt, count) in enumerate(top_prompts.items(), 1):
    print(f"Rank {i}: {count} samples ({round(100 * count/len(eng_battles_no_ties), 2)}%)\n{prompt}\n")

# print the total percentage of prompts that are 1 of the top 10 prompts
top_10_percentage = sum(top_prompts) / len(eng_battles_no_ties)
print(f"Total percentage of prompts that are 1 of the top 10 prompts: {round(100 * top_10_percentage, 2)}%")

## Question 3a: Measure the Effect of Frequent Prompts

When evaluating models, it is useful to understand which prompt types are over-represented and how these popular prompts can influence the leaderboard. To isolate the effect of removing the top 10 prompts, both leaderboards below use the same English-or-unknown-language battle subset, but one leaderboard computes pairwise win rates with the frequent prompts included and the second leaderboard computes pairwise win rates with frequent prompts removed.


In [ ]:
# Run this cell, no code changes required :)

# Build the baseline on the English-or-unknown-language subset.
pairwise_win_rate_eng = get_pairwise_win_rate(eng_battles_no_ties)
avg_win_fig_eng = get_pairwise_win_rate_plot(
    pairwise_win_rate_eng,
    title="Average Win Rate Against All Other Models (English/Unknown, All Prompts)",
)

# Remove the top prompts from that same subset.
eng_battles_no_ties_no_top_prompts = eng_battles_no_ties[
    ~eng_battles_no_ties["prompt"].isin(top_prompts.index)
]
pairwise_win_rate_no_top_prompts = get_pairwise_win_rate(
    eng_battles_no_ties_no_top_prompts
)
avg_win_fig_no_top_prompts = get_pairwise_win_rate_plot(
    pairwise_win_rate_no_top_prompts,
    title="Average Win Rate Against All Other Models (No Top Prompts)",
)

if not IS_GRADING_ENV:
    avg_win_fig_eng.show()
    avg_win_fig_no_top_prompts.show()


**Question:** Compare the English/unknown-language leaderboard with all prompts (`pairwise_win_rate_eng`) to the leaderboard from the same battles after the top 10 prompts are removed (`pairwise_win_rate_no_top_prompts`). Is there a difference between the two leaderboards? List any models in the top 5 that have changed position.


Write your response in `q3ai_answer` and run the answer cell to save it for the written submission.


<!-- BEGIN QUESTION -->



In [ ]:
# TODO: Write your answer in `q3ai_answer`, then run this cell to save it.
q3ai_answer = """
YOUR ANSWER HERE
"""
save_written_answer(q3ai_answer, "q3ai")


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Question:** What is a potential reason that this leaderboard has shifted? We are looking for an answer which relates to potential model behavior, not just "prompts were removed thus the leaderboard changed"


Write your response in `q3aii_answer` and run the answer cell to save it for the written submission.

In [ ]:
# TODO: Write your answer in `q3aii_answer`, then run this cell to save it.
q3aii_answer = """
YOUR ANSWER HERE
"""
save_written_answer(q3aii_answer, "q3aii")


<!-- END QUESTION -->

## Question 3b: Compare Rankings Across Prompt Categories

LMArena also provides more detailed **category labels**, in the `is_code` column and the
nested `category_tag` column. We have already extracted the boolean Series below for you —
feel free to reuse them when you analyze your clusters in Question 3c.

Use the following category names as keys in both `proportions` and `category_dataframes_map`, and as the labels in the `category` column of `rank_dataframes`. Note that we use `instruction_following` for clarity in this assignment, while the dataset uses `if` for that field :)


| Required category name | Provided boolean Series | Dataset field |
| --- | --- | --- |
| `creativity` | `expected_creativity` | `category_tag['criteria_v0.1']['creativity']` |
| `technical_accuracy` | `expected_technical_accuracy` | `category_tag['criteria_v0.1']['technical_accuracy']` |
| `instruction_following` | `expected_instruction_following` | `category_tag['if_v0.1']['if']` |
| `math` | `expected_math` | `category_tag['math_v0.1']['math']` |
| `is_code` | `expected_is_code` | `is_code` |


1. For each category, compute the fraction of battles where it is `True`, and plot the five proportions as a bar chart. Keep the `save_figure(prompt_proportions_bar_plot, "q3bi")` line so your figure is written to `written/q3bi.png`.

2. Use the staff-provided helper `get_pairwise_win_rate` (from Question 2b) to compute win rates and ranks separately for each category's subset of battles.

3. Combine those results into one DataFrame in *tidy format* — one row per (model, category) pair, with `model`, `category`, and `rank` columns — and pass it to `plot_category_rank_heatmap`. Include the five categories above plus an `'overall'` category holding the `pairwise_win_rate` you computed earlier. With 20 models and 6 categories, you should end up with 120 rows.

You might notice that the leaderboards can change quite a bit! This is because different model developers often put more emphasis on ceratin tasks in training their LLMs to better cater to their audience. Many of these models are also limited by the amount of data and compute available, which can further force specialization as some tasks are much harder to learn (especially if the model is on the smaller side). 

In [ ]:
def plot_category_rank_heatmap(df: pd.DataFrame) -> go.Figure:
    """
    Staff-provided helper!
    
    Create and return a heatmap of model ranks by category.
    """
    assert "overall" in df["category"].unique(), \
        "'overall' was not found as a category in your dataframe"

    rank_table = df.pivot(
        index="model",
        columns="category",
        values="rank"
    )

    if "overall" in rank_table.columns:
        cols = ["overall"] + [
            c for c in rank_table.columns if c != "overall"
        ]
        rank_table = rank_table[cols]

    rank_table = rank_table.sort_values("overall", ascending=True)

    model_ranks_by_cat_heatmap_fig = px.imshow(
        rank_table,
        text_auto=True,
        color_continuous_scale="Viridis_r",
        labels=dict(x="Category", y="Model", color="Rank (1=best)"),
        zmin=1,
        zmax=rank_table.max().max(),
        aspect="auto"
    )

    model_ranks_by_cat_heatmap_fig.update_layout(
        title="Overall and Per-Category Ranks",
        width=950,
        height=400 + 12 * len(rank_table),
        xaxis_side="top"
    )

    return model_ranks_by_cat_heatmap_fig

In [ ]:
# GIVEN BOOLEAN SERIES (do not modify)
expected_creativity = eng_battles_no_ties['category_tag'].apply(lambda x: x['criteria_v0.1']['creativity'])
expected_technical_accuracy = eng_battles_no_ties['category_tag'].apply(lambda x: x['criteria_v0.1']['technical_accuracy'])
expected_instruction_following = eng_battles_no_ties['category_tag'].apply(lambda x: x['if_v0.1']['if'])
expected_math = eng_battles_no_ties['category_tag'].apply(lambda x: x['math_v0.1']['math'])
expected_is_code = (eng_battles_no_ties['is_code'] == True)

# TODO: build `proportions`, a dict mapping each category name to the fraction of
# battles where that category is True, then plot it as a bar chart.
# Use these keys: 'creativity', 'technical_accuracy',
# 'instruction_following', 'math', 'is_code' (see the table above).
proportions = ...
prompt_proportions_bar_plot = ...
save_figure(prompt_proportions_bar_plot, "q3bi")

# TODO: build `category_dataframes_map`, a dict mapping each category name to the
# subset of `eng_battles_no_ties` where that category is True.
# Use the same five keys as in `proportions`.
category_dataframes_map = ...

# TODO: call get_pairwise_win_rate on each subset, tag it with its category name, and
# stack everything into one tidy DataFrame: one row per (model, category), with
# `model`, `category`, and `rank` columns. The 'overall' rows are tagged for you below.
pairwise_win_rate['category'] = 'overall'
rank_dataframes = [pairwise_win_rate]
for category_name, category_df in category_dataframes_map.items():
    ...

rank_dataframes = ...

if not IS_GRADING_ENV:
    model_ranks_by_cat_heatmap_fig = plot_category_rank_heatmap(rank_dataframes)
    prompt_proportions_bar_plot.show()
    model_ranks_by_cat_heatmap_fig.show()
save_figure(model_ranks_by_cat_heatmap_fig, "q3bii")

In [ ]:
grader.check("q3b")

## Question 3c: Reduce Sparse Text Features with LSA

The category labels in Question 3b give us a clean way to slice the data, but somebody had to decide on those five categories in advance. If the interesting structure in the prompts is something nobody thought to label, those categories cannot reveal it. In this question we let the data suggest its own grouping, using two tools from lecture: **PCA** for dimensionality reduction and a **Gaussian mixture model** for clustering.

We begin from a TF-IDF representation of the prompt text. TF-IDF produces a very wide and very sparse matrix, with thousands of columns of which almost all are zero on any given row. This representation is unsuitable for a Gaussian mixture model for two separate reasons.

- It's intractable to fit a GMM directly on TF-IDF's outputs! A Gaussian mixture estimates a mean and a covariance per component over dense vectors. With thousands of features, a full covariance matrix has millions of free parameters, and there is nowhere near enough data to estimate them.
- Distances stop being informative. In very high dimensions almost every pair of sparse documents shares little vocabulary, so their distances are all similar and any distance-based method degrades.

To solve both problems, we can first use PCA to reduce the TF-IDF representations into a lower-dimensional space. This reduction is necessary before fitting the GMM. Estimating a Gaussian distribution over thousands of TF-IDF features would require too many parameters and too much data. Reducing the prompts to a much smaller dense representation makes the GMM practical to fit while retaining important patterns in the prompt text. This pipeline of TF-IDF followed by a truncated SVD is common enough to have its own name, **latent semantic analysis (LSA)**. Each component is a direction in vocabulary space capturing a group of words that tend to co-occur, and a prompt's coordinates record how much of each word group it uses.

**What exactly is a truncated SVD?**

Like PCA, Truncated SVD is also a dimensionality-reduction method. It finds a small set of directions that capture important patterns in the original data, then represents each prompt using only those directions. For example, it can turn a prompt represented by thousands of individual word features into a vector of just 20–50 values. These new values summarize broader patterns of words that tend to appear together.

We use TruncatedSVD instead of ordinary PCA because TF-IDF produces a sparse matrix: most of its entries are zero. Ordinary PCA first subtracts each column’s mean, which would turn many of those zeros into nonzero values and create a large, dense matrix. TruncatedSVD works directly with the sparse TF-IDF matrix, making it much more memory-efficient.

---

We've provided the code to fit a TF-IDF vectorizer on the prompt texts. Your task is to:
1. Initialize a `TruncatedSVD` model ([scikit-learn docs](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)). Make sure to set `N_SVD_COMPONENTS` as desired output dimensions and `SEED` as the random seed when initializing!
2. Fit the `TruncatedSVD` model to the TF-IDF matrix `X_tfidf`.

In [ ]:
# Staff-provided setup. Run this cell without changes :)
eng_battles_sample = eng_battles_no_ties_no_top_prompts.sample(n=8000, random_state=SEED).copy()
prompt_texts = eng_battles_sample["prompt"].to_numpy()

tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(prompt_texts)

density = 100 * X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print(f"TF-IDF shape: {X_tfidf.shape}; only {density:.2f}% of entries are nonzero.")


In [ ]:
N_SVD_COMPONENTS = 20

# TODO: construct a TruncatedSVD with N_SVD_COMPONENTS components and random_state=SEED
svd = ...

# TODO: fit it on X_tfidf and transform the prompts
X_reduced = ...

print("Reduced feature shape:", X_reduced.shape)
print(f"Variance retained by {N_SVD_COMPONENTS} components: "
      f"{svd.explained_variance_ratio_.sum():.1%}")


In [ ]:
grader.check("q3c")

## Question 3d: Fit a Gaussian Mixture Model

A GMM is a weighted collection of Gaussian components with an unobserved assignment variable $z$. Unlike K-Means, which gives each prompt one hard cluster label, a GMM gives a probability for every component:

$$P(z=k \mid x).$$

These probabilities are called **responsibilities** or **soft-assignments**.

Fit a five-component GMM to the reduced prompt features, then store both the hard assignments and the responsibilities.


When creating your GMM:
- Set `covariance_type="diag"` when instantiating your `GaussianMixture` model.
- Make sure to also set `n_components` equal to `N_COMPONENTS`. This parameter controls how many Gaussian distributions we want to include in our GMM.
- Also make sure to set the `random_state = SEED` parameter for reproducibility!

**Hints**:
- [Scikit-learn docs on `GaussianMixture`](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html)


In [ ]:
N_COMPONENTS = 5

# TODO: construct the GaussianMixture
gmm = ...

# TODO: fit the GMM and obtain one hard component assignment per prompt
gmm_labels = ...

# TODO: obtain the responsibility probabilities for every prompt
responsibilities = ...

eng_battles_sample["cluster"] = gmm_labels

# Given: a K-Means baseline and a two-dimensional view of the GMM assignments.
kmeans = KMeans(n_clusters=N_COMPONENTS, random_state=SEED, n_init=10).fit(X_reduced)
cluster_plot_df = pd.DataFrame({
    "Component 1": X_reduced[:, 0],
    "Component 2": X_reduced[:, 1],
    "GMM component": eng_battles_sample["cluster"].astype(str).to_numpy(),
})
clusters_2d_plot = px.scatter(
    cluster_plot_df,
    x="Component 1",
    y="Component 2",
    color="GMM component",
    opacity=0.6,
    title="GMM Prompt Components on the First Two SVD Dimensions",
)
clusters_2d_plot.update_traces(marker={"size": 4})
clusters_2d_plot.update_layout(width=800, height=600)
if not IS_GRADING_ENV:
    clusters_2d_plot.show()
save_figure(clusters_2d_plot, "q3di")


In [ ]:
grader.check("q3d")

## Question 3e: Interpret Soft Assignments and Discovered Topics

The following staff-provided cells translate each component mean back into vocabulary terms, compare the GMM's hard labels with K-Means, and show prompts for which the GMM is least confident. They then reuse the Question 3b heatmap to compare model rankings across the five discovered topics.


In [ ]:
# Staff-provided helper. Run this cell without changes :)
def top_terms_per_cluster(gmm, svd, vectorizer, n_terms=10):
    term_weights = svd.inverse_transform(gmm.means_)
    vocabulary = np.array(vectorizer.get_feature_names_out())
    return {
        cluster: vocabulary[np.argsort(term_weights[cluster])[::-1][:n_terms]].tolist()
        for cluster in range(gmm.n_components)
    }

cluster_terms = top_terms_per_cluster(gmm, svd, tfidf_vectorizer)
cluster_labels = {
    cluster: f"c{cluster}: " + ", ".join(terms[:3])
    for cluster, terms in cluster_terms.items()
}

for cluster, terms in cluster_terms.items():
    cluster_rows = eng_battles_sample[eng_battles_sample["cluster"] == cluster]
    print(f"Component {cluster} ({len(cluster_rows)} prompts)")
    print("  top terms:", ", ".join(terms))
    if len(cluster_rows):
        example = cluster_rows["prompt"].iloc[0]
        print("  example:", example[:160].replace("\n", " "))


In [ ]:
# Staff-provided comparison. Run this cell without changes.
agreement = pd.crosstab(
    eng_battles_sample["cluster"],
    kmeans.labels_,
    rownames=["GMM component"],
    colnames=["K-Means cluster"],
)
print("GMM vs. K-Means hard assignments:")
display(agreement)

# A low maximum responsibility means the GMM is uncertain about the hard label.
most_ambiguous = np.argsort(responsibilities.max(axis=1))[:5]
print("\nFive prompts with the least certain GMM assignments:\n")
for position in most_ambiguous:
    weights = responsibilities[position]
    top_two = np.argsort(weights)[::-1][:2]
    prompt = prompt_texts[position][:160].replace("\n", " ")
    print(prompt)
    print(f"  component {top_two[0]}: {weights[top_two[0]]:.2f}; "
          f"component {top_two[1]}: {weights[top_two[1]]:.2f}\n")


In [ ]:
# Staff-provided leaderboard comparison. Run this cell without changes.
clustered_battles = eng_battles_no_ties_no_top_prompts.copy()
all_prompt_features = tfidf_vectorizer.transform(clustered_battles["prompt"].to_numpy())
clustered_battles["cluster"] = gmm.predict(svd.transform(all_prompt_features))

overall_ranks = pairwise_win_rate_no_top_prompts.copy()
overall_ranks["category"] = "overall"
cluster_rank_frames = [overall_ranks]
for cluster, cluster_battles in clustered_battles.groupby("cluster"):
    cluster_ranks = get_pairwise_win_rate(cluster_battles)
    cluster_ranks["category"] = cluster_labels[cluster]
    cluster_rank_frames.append(cluster_ranks)

cluster_rank_dataframe = pd.concat(cluster_rank_frames, ignore_index=True)
cluster_rank_heatmap_fig = plot_category_rank_heatmap(cluster_rank_dataframe)
cluster_rank_heatmap_fig.update_layout(title="Model Ranks Within Discovered Prompt Topics")
save_figure(cluster_rank_heatmap_fig, "q3ei")

print("Battles per component:")
print(clustered_battles["cluster"].value_counts().sort_index().to_string())
if not IS_GRADING_ENV:
    cluster_rank_heatmap_fig.show()


<!-- BEGIN QUESTION -->

Answer both parts using the output above.

**Questions:**
1. Choose one ambiguous prompt. Explain what its two largest responsibilities/soft-assignments say that a single hard K-Means label cannot express.
2. Give short names to at least two GMM components using their top terms and examples.

Write your response in `q3eii_answer` and run the answer cell to save it.


In [ ]:
# TODO: Write your answer in q3eii_answer, then run this cell to save it.
q3eii_answer = """
1. YOUR ANSWER HERE

2. YOUR ANSWER HERE
"""
save_written_answer(q3eii_answer, "q3eii")


<!-- END QUESTION -->

# Question 4: Learning Model Strengths

Average model win-rate is not an ideal metric for comparing models when battle counts per model are not equal. For instance, if ChatGPT-4o-latest battled more often with weaker models, it would have a high win rate without being an actually stronger model. Now let's explore how we can instead *learn* these model strengths.

**Goal:** we want to construct a leaderboard by assigning a strength score $S_m$ to each model $m \in \{1,...,M\}$, such that:
- The ranking reflects the probability of one model winning against another.
- For any pair of models A and B, the probability that A beats B, should depend on the *difference* in their strengths: $S_A - S_B$. Why the difference? Since we are measuring pairwise preference, there is no absolute measure of strength but rather a model's strength *relative* to other models.

**Formally, we want a function $f$ such that**
- For models A and B with scores $S_A$ and $S_B$, we want:
  $$ P(\text{A beats B}) = f(S_A - S_B) $$
- The function $f$ should be be positively correlated with how much "better" model A is compared to model B (bigger skill gap, higher win chance), and always output a probability between 0 and 1.

At this point, a natural question is: what should we choose for the function $f$? A standard and effective choice is the logistic (sigmoid) function:

$$ P(\text{A beats B}) = \sigma(S_A - S_B) = \frac{1}{1 + e^{-(S_A - S_B)}} $$

Notice that this is exactly the same form as logistic regression, where the model scores are the parameters to be learned. In other words, learning model strengths from pairwise outcomes is equivalent to fitting a logistic regression model to the data.

So, we can use logistic regression to learn the model strengths that best explain the observed battle outcomes. The higher a model's score, the more likely it is to win against others. The methodology is called the [Bradley-Terry](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) model and is the underlying theory to other common scoring systems like ELO ratings.

---

## Representing battles as features

To learn model strengths, we first need to represent each battle in a form that logistic regression can use.

Each battle involves two models, which we call Model A and Model B. We already remove tied battles so that every remaining battle has a winner.

Suppose there are $M$ total models. We create an $M$-dimensional feature vector for each training example:

- Each coordinate corresponds to one model.
- The model represented as Model A receives a `+1` in its coordinate.
- The model represented as Model B receives a `-1` in its coordinate.
- Every model not involved in the battle receives a `0` in their coordinates.

The label is a single binary value:

- The label is `1` if the model represented by `+1` won.
- The label is `0` if the model represented by `+1` lost.

The label does not identify a model. The feature vector identifies the models involved, while the label describes the outcome from the perspective of the `+1` model.

### Creating two training examples

We can create two training examples from each battle. The first example treats Model A as the `+1` model and Model B as the `-1` model. Its label indicates whether Model A won. The second example reverses the roles: Model B becomes the `+1` model and Model A becomes the `-1` model. Its label indicates whether Model B won.

For example:

```python
row = {
    "model_a": "gpt-4o-2024-05-13",
    "model_b": "claude-3-opus-20240229",
    "winner": "model_a"
}
```

Assume we are learning the strengths of four models, with the following coordinate assignments:

```text
index 0: gpt-4o
index 1: claude-3-opus
index 2: llama
index 3: gemini
```

This battle produces the following two training examples:

**Training example 1**

```text
Feature vector: [ 1, -1,  0,  0]
Label: 1
```

* GPT-4o receives `+1`.
* Claude receives `-1`.
* the other models receive `0` because they were not involved in this battle.
* The label is `1` because GPT-4o, the `+1` model, won.

**Training example 2**

```text
Feature vector: [-1,  1,  0,  0]
Label: 0
```

* GPT-4o receives `-1`.
* Claude receives `+1`.
* the other models receive `0` because they were not involved in this battle.
* The label is `0` because Claude, the `+1` model, lost.

The two feature vectors are negatives of each other. This allows the same battle to be represented from both perspectives.

### Why use this representation?

Let $S_i$ denote the learned strength of model $i$. For the first feature vector, the logistic regression score is

$$ S_{\text{GPT-4o}}  - S_{\text{Claude}}$$

Therefore, taking the sigmoid of this score difference results in $\sigma(S_{\text{GPT-4o}}  - S_{\text{Claude}})$, which equals the probability that GPT-4o beat Claude: $P(\text{GPT-4o beats Claude-3-opus})$.


For the reversed feature vector, the score is

$$ S_{\text{Claude}}  - S_{\text{GPT-4o}}$$

and taking the sigmoid gives $\sigma(S_{\text{Claude}}  - S_{\text{GPT-4o}})$, which equals the probability that Claude beat GPT-4o: $P(\text{Claude-3-opus beats GPT-4o})$.

This representation lets logistic regression learn one strength coefficient per model and compare two models by taking the difference between their strengths.

---

## Constructing the feature matrix and labels

Now we generalize the representation from one pair of models and one row in our dataset to all models in the dataset.

Suppose there are $M$ models and $B$ non-tied battles. Each model has an unknown strength, represented by a scalar $S_i$. We collect all model strengths into the strength vector

$$
\mathbf{S} =
\begin{bmatrix}
S_1 \\
S_2 \\
\vdots \\
S_M
\end{bmatrix}
\in \mathbb{R}^{M}.
$$

The position of each strength in $\mathbf{S}$ corresponds to the same model position used in every feature vector. For example, if the models are ordered as `[gpt-4o, claude, llama, gemini]`, then

$$
\mathbf{S} =
\begin{bmatrix}
S_{\text{gpt-4o}} \\
S_{\text{claude}} \\
S_{\text{llama}} \\
S_{\text{gemini}}
\end{bmatrix}.
$$

For every battle, we create two training examples: one from Model A's perspective and one from Model B's perspective. Therefore, $B$ original battles produce $2B$ training examples.

We organize the $2B$ feature vectors into a feature matrix

$$
\mathbf{X} \in \mathbb{R}^{2B \times M}.
$$

Each row of $\mathbf{X}$ represents one oriented comparison:

- the model treated as the potential winner receives $+1$;
- its opponent receives $-1$;
- every other model receives $0$.

For example, if the model order is $[A,B,C]$, then a row representing “A versus C, with A as the potential winner” is

$$
[1, 0, -1].
$$

The observed outcomes are collected into a label vector $\mathbf{y}$, of dimension $2B$. Each row of $\mathbf{X}$ correspondes to a coordinate in $\mathbf{y}$:

- $y=1$ if the model represented by $+1$ won;
- $y=0$ if the model represented by $+1$ lost.

Going back to the previous example, if A beats C, the two training examples are

$$
\mathbf{x}_1 = [1,0,-1],
\qquad y_1=1,
$$

and

$$
\mathbf{x}_2 = [-1,0,1],
\qquad y_2=0.
$$

The logistic regression model computes one score for each row of $\mathbf{X}$. These scores are obtained by multiplying the feature matrix by the strength vector. This produces another vector of dimension $2B$:

$$
\mathbf{X}\mathbf{S} \in \mathbb{R}^{2B}.
$$

For a row representing A versus C, the score produced after multiplying $\mathbf{X} \mathbf{S}$ is

$$
S_A-S_C.
$$

For the reversed row, the score is

$$
S_C-S_A.
$$

Applying the sigmoid function element-by-element converts these scores into predicted win probabilities:

$$
\hat{\mathbf{y}}
=
\sigma(\mathbf{X}\mathbf{S}).
$$

Thus, $\hat{\mathbf{y}}$ has one predicted probability for every training example. Its dimensions are

$$
\hat{\mathbf{y}} \in \mathbb{R}^{2B}.
$$

The model learns the strength vector $\mathbf{S}$ so that these predicted probabilities match the observed labels $\mathbf{y}$ as closely as possible.

In summary:

- $\mathbf{X}$ has $2B$ rows and $M$ columns.
- Each row of $\mathbf{X}$ describes **who played whom** and in what direction.
- $\mathbf{S}$ has one strength value $S_i$ for each model. These are the strengths we are trying to learn.
- $\mathbf{y}$ contains the observed win/loss labels.
- $\hat{\mathbf{y}} = \sigma(\mathbf{X} \cdot \mathbf{S})$ gives us the predicted win probabilities.

## Question 4a: Represent Battles as Features

In order to train our logistic regression model which will learn the chatbot model strengths, we need to featurize all the battles we have.

Implement the function below to transform `selected_battles_no_ties` and `selected_lmarena_models` into feature vectors and labels. This will allow us to represent each battle as input-output pairs for training.

In [ ]:
def turn_into_features(df, selected_models):
    '''
    Convert pairwise battle results into feature matrix X and label vector y
    suitable for logistic regression based on the Bradley-Terry model
    '''
    # TODO:
    # 1. Initialize lists X and y, then iterate through each row in the DataFrame.
    # 2. For each battle, append the A-to-B training example:
    #    - The feature vector has +1 for 'model_a', -1 for 'model_b', and 0 elsewhere.
    #    - The label is 1 if 'model_a' won and 0 if 'model_b' won.
    # 3. Append the reverse B-to-A training example for the same battle:
    #    - Negate the A-to-B feature vector, giving +1 to 'model_b' and -1 to 'model_a'.
    #    - Use the complementary label: 1 if 'model_b' won and 0 if 'model_a' won.
    # 4. Each battle should therefore contribute two rows to X and two entries to y.
    # 5. Return the feature matrix X and label vector y as NumPy arrays.
    ...
    return np.array(X), np.array(y)

X, y = turn_into_features(selected_battles_no_ties, selected_lmarena_models)
X.shape, y.shape

In [ ]:
grader.check("q4a")

## **Question 4b: Fit the Bradley-Terry Model**
Now that we have extracted the features from the previous question, we can fit the logistic regression model and learn one strength coefficient per LLM.

Use `LogisticRegression(fit_intercept=False, penalty="l2", C=1.0, tol=1e-6)`, fit it on `X` and `y`, and store the learned strengths in `results_df`, sorted from highest to lowest score. The provided cell also copies this table to `q4b_results_df`; keep that name because the tests use it to check your result.

The derivation above describes the unregularized Bradley-Terry maximum-likelihood objective. Here we follow the practical LMArena-style implementation and use scikit-learn's L2 penalty with `C=1.0`. The resulting coefficients are therefore **regularized (shrunk) Bradley-Terry estimates**, not the unregularized MLE. The reason we use regularization in practice is because some model pairs have relatively few battles or very one-sided outcomes.

These skewed battle results which can make unregularized strength estimates unstable or extremely large. L2 regularization gently pulls coefficients toward zero, producing a more stable leaderboard that is less sensitive to sampling noise.


In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: Train the model with the features and labels created in Question 4a
lr = LogisticRegression(
    fit_intercept=False, penalty="l2", C=1.0, tol=1e-6
)
lr.fit(...)
scores = ...

results = {"Model": selected_lmarena_models, "Score": scores}
results_df = pd.DataFrame(results).sort_values("Score", ascending=False).reset_index(drop=True)
q4b_results_df = results_df.copy()
results_df

In [ ]:
grader.check("q4b")

## Question 4c: Evaluate Win-Probability Predictions on Held-Out Battles

A leaderboard fit can describe the battles it was trained on, but that does not tell us how well it predicts new battles. We will therefore reserve 20% of the battles as a test set and compare two predictors:

- **Bradley–Terry:** probabilities from a logistic regression fit only on the training battles.
- **Pairwise win-rate baseline:** the empirical training-set win fraction for that exact pair of models. If the pair never occurred in training, predict 0.5.

Complete the scaffold below, keeping the provided variable names and table labels:

1. **Split the battles into a training and test set:** Use `train_test_split` on `selected_battles_no_ties` with `test_size=0.2`, `random_state=SEED`, and `stratify=selected_battles_no_ties["winner"]`. Store them into the two DataFrames `train_battles` and `test_battles`.
    a. Preserve the original ordering of the rows! For example, a battle originally at index `42` should still have index `42` in whichever split contains it. Passing the DataFrame directly to `train_test_split` automatically preserves its original index labels, just make sure not to call `reset_index()` or assign a new `.index`.
    b. Make sure to split **before** calling `turn_into_features`! Each battle creates two reversed feature rows, so splitting the expanded rows after turning them into features could leak the same outcome across training and testing.
2. **Fit Bradley–Terry on training data only:** Call `turn_into_features` separately on `train_battles` and `test_battles`, passing `selected_lmarena_models` as the second argument to both calls (recall that `turn_into_features` creates one column for each model in the list of models passed in). Then fit the logistic regression model `bt_lr` on `X_train` and `y_train`.
3. **Predict each held-out battle once:**

    - **Create the labels:** Create a one-dimensional NumPy array called `test_labels` containing 1 if Model A won, 0 if Model B won, in `test_battles` row order.
    - **Predict win probabilities:** Create a one-dimensional NumPy array called `bt_probs` containing one Model-A win probability per test battle, in the same order as `test_labels`. Recall that `turn_into_features` creates two rows per battle: one for A versus B, and a reversed copy for B versus A with the feature signs and label flipped. Use only the A-versus-B rows so each battle is evaluated once and the predictions match `test_labels`. Use `bt_lr.predict_proba` and select the probabilities the LR model outputs for class 1 (class 1 is the class it predicts when model A wins). Since ties are excluded, Model B's win probability is `1 - bt_probs`.

4. **Build the baseline:** Compute training win fractions with `compute_pairwise_win_fraction` and reindex the table axes so that they match the same row-ordering `selected_lmarena_models`. Create a one-dimensional NumPy array called `baseline_probs`: for each test battle, use Model A's training win fraction against Model B, in `test_labels` order.

    - **Unseen pairs:** Two models may face each other in a test battle without having faced each other in the training data. Their training win fraction is missing (`NaN`). You should replace missing training win fractions with 0.5 to express no preference.
    - **Clipping:** Clip probabilities to `[1e-6, 1 - 1e-6]`. We'll be taking the logarithm of these probabilities later, and we want to avoid accidentally taking the log of zero (which results in negative infinity).

5. **Evaluate both methods!** Compare `bt_probs` and `baseline_probs` against `test_labels`.

    Scikit-learn provides two scoring functions, already imported from `sklearn.metrics` in the starter cell. You can call them directly:

    - **`accuracy_score(y_true, y_pred)`** returns the fraction of predictions that match the true labels. Pass `test_labels` as `y_true` and predicted 0/1 labels as `y_pred`. Since the LR model we trained predicts **probabilities** rather than 0/1 binary classes, use a threshold of 0.5 (meaning that we count the LR model's prediction as a 1 when the probability it predicted is at least 0.5). Generally, a higher accuracy score is better!
    - **`log_loss(y_true, y_pred)`** returns mean binary cross-entropy. Pass `test_labels` as `y_true` and the class-1 probabilities as `y_pred`, without thresholding them. Lower is better; confident wrong predictions receive a larger penalty.

Create a pandas DataFrame called `q4c_metrics` with shape **`(2, 3)`**: one row for each prediction method, and three columns called **`Method`, `Accuracy`, `Log Loss`**, in that order. Each row summarizes a method's performance across all test battles. The first row should contain the `Bradley-Terry` results, and the second should contain the `Pairwise win-rate baseline` results. These labels are filled in for you below; complete the metric values.

For example, the completed table might look like this (these numbers are illustrative, not expected answers):

| Method | Accuracy | Log Loss |
| --- | --- | --- |
| Bradley-Terry | 0.65 | 0.62 |
| Pairwise win-rate baseline | 0.60 | 0.68 |

In this example, Bradley–Terry predicts 65% of test winners correctly with mean cross-entropy 0.62; the pairwise baseline predicts 60% correctly with mean cross-entropy 0.68.


In [ ]:
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split

# 1. Split the battles into a training and test set.
# Split before calling turn_into_features.
# Use test_size=0.2, random_state=SEED, and stratify by winner.
# Keep these DataFrame names. Passing the DataFrame directly preserves its index
# labels automatically; do not reset_index() or assign a new .index afterward.
train_battles, test_battles = train_test_split(
    selected_battles_no_ties,
    test_size=...,
    random_state=...,
    stratify=...,
)

# 2. Fit Bradley-Terry on training data only.
# Convert each split into features with the Q4a helper.
# Use the same model list so column j represents the same model in both splits.
X_train, y_train = turn_into_features(..., selected_lmarena_models)  # Training battles
X_test, _ = turn_into_features(..., selected_lmarena_models)  # Test battles
# Fit a fresh model on training features and labels only.
bt_lr = LogisticRegression(
    fit_intercept=False, penalty="l2", C=1.0, tol=1e-6
)
bt_lr.fit(...)

# 3a. Create a one-dimensional NumPy array called test_labels based on test_battles
# Use 1 if Model A won and 0 if Model B won.
test_labels = ...

# 3b. Create a one-dimensional NumPy array called bt_probs, in test_labels order.
# Recall that each battle has an A-versus-B row and a reversed B-versus-A row;
# Use only the A-versus-B rows to evaluate each battle once.
# Use bt_lr.predict_proba and select class-1 (Model A wins) probabilities.
bt_probs = ...

# 4. Build the baseline!
# Compute win fractions from training battles only
train_pairwise = compute_pairwise_win_fraction(...).reindex(
    index=selected_lmarena_models, columns=selected_lmarena_models
)
baseline_probs = np.array([
    ...  # Look up the win fraction for row.model_a against row.model_b
    for row in test_battles.itertuples()
])
# Unseen pairs: replace missing training win fractions with 0.5
baseline_probs = ...
# Clipping: keep probabilities within [1e-6, 1 - 1e-6]
baseline_probs = ...

# 5. Evaluate both methods and store the results as a (2, 3) DataFrame
# Compare bt_probs and baseline_probs against test_labels:
# - Accuracy: use accuracy_score with predictions thresholded at >= 0.5.
# - Log Loss: use log_loss with probabilities to compute mean binary cross-entropy.
q4c_metrics = pd.DataFrame({
    "Method": ["Bradley-Terry", "Pairwise win-rate baseline"],
    "Accuracy": [
        ...,  # Bradley-Terry accuracy
        ...,  # Pairwise baseline accuracy
    ],
    "Log Loss": [
        ...,  # Bradley-Terry cross-entropy
        ...,  # Pairwise baseline cross-entropy
    ],
})

q4c_metrics

In [ ]:
grader.check("q4c")

<!-- BEGIN QUESTION -->

## Question 4d: Interpret the Held-Out Evaluation

Using your `q4c_metrics` results from Question 4c, briefly state which method has the lower held-out log loss and explain why held-out performance is more informative than training performance.

Write your response in `q4d_answer` and run the answer cell to save it for the written submission.

In [ ]:
# TODO: Write your answer in `q4d_answer`, then run this cell to save it.
q4d_answer = """
YOUR ANSWER HERE
"""
save_written_answer(q4d_answer, "q4d")


<!-- END QUESTION -->

# Question 5: Quantifying Ranking Uncertainty

## Confidence Intervals

From the previous question, we were able to train the LR model and obtain the scores $S_i$ of the large language models from LMArena. 

However, when comparing model strength scores, it's important to understand not just the average performance, but also how much uncertainty there is in our estimates. Our rankings are based on a finite sample of battles, and if we had collected a different set of match-ups, the resulting scores could be different. This sampling variability means that our estimated model strengths are subject to noise.

Bootstrapping is a powerful, intuitive way to assess this uncertainty without making strong assumptions about the underlying data. By repeatedly resampling our observed battles (with replacement) and retraining the LR model on each resampled dataset, we simulate what might have happened if we had observed a slightly different set of battles. For each resample, the LR model will produce a new set of model scores. By looking at the distribution of these bootstrapped scores, we can estimate confidence intervals for each LLM's strength.

This notebook uses 200 bootstrap replicates. One replicate is one simulated rerun of the analysis: sample rows from the observed data with replacement, refit the model, and save its coefficients. In a real statistical analysis, you would generally use at least 1,000 replicates and check that the reported intervals are stable as the number of replicates increases. We use 200 here to keep the homework runtime manageable.

In short, bootstrapping helps us answer: "If we repeated this evaluation process many times, how much could each large language model's score (as predicted by the trained logistic regression model) vary just due to random chance in which battles we happened to observe?" This gives us a more honest sense of which differences in model scores are robust, and which might just be due to luck.

## Question 5a: Bootstrap Model Strengths

Let's implement a function that returns the learned scores from repeatedly training a logistic regression model on different samples of our dataset and confidence intervals after bootstrapping.

1. Each bootstrap replicate is one resampling-and-fitting run.
    - Randomly draw `len(X)` indices from `0` through `len(X) - 1` **with replacement**, so an index may appear multiple times and some indices may not appear at all.
    - Use the **same sampled indices, in the same order**, to form `X_boot` from the rows of `X` and `y_boot` from the entries of `y`. Using the same indices to extract from both `X` and `y` ensures each feature row is in the same order as its label. The resampled dataset has the same number of rows as `X`.
    - Fit a fresh logistic regression model on `X_boot` and `y_boot` using the settings from Question 4b. Repeat this process `n_bootstrap` times, drawing a new sample each time.

2. Before the bootstrap loop, create an empty list called `bootstrap_coefficients`. Each time you fit a logistic regression model, its coefficient vector `lr.coef_[0]` contains one learned strength for each feature and the coefficients are in the same order as `feature_names`. Append that entire vector to `bootstrap_coefficients` for each bootstrap replicate.
3. After all `n_bootstrap` models have been fit, convert the list to a NumPy array. If X has `n_features = X.shape[1]`, the resulting array should have shape `(n_bootstrap, n_features)`. Each row contains all coefficients from one fitted model, and each column contains one feature's coefficient across all replicates. Thus, `bootstrap_coefficients[b, j]` is the strength learned for `feature_names[j]` in replicate `b`.
4. Use the bootstrap coefficient array `bootstrap_coefficients` to compute a 95% percentile interval for each feature. For each feature separately, consider its coefficient across all bootstrap replicates and find the 2.5th and 97.5th percentiles with `np.percentile`. Store the result in `confidence_intervals`, an array of shape `(2, n_features)`: `confidence_intervals[0, j]` is the 2.5th-percentile lower bound for `feature_names[j]`, and `confidence_intervals[1, j]` is the 97.5th-percentile upper bound. These bounds are the actual coefficient values, not distances from the mean.
5. Compute `mean_scores`, the mean bootstrapped strength for each feature. It should be an array of shape `(n_features,)` in the same order as `feature_names`.
6. Build `results_df` with columns `Feature`, `Category`, `Average Score`, `Lower Bound`, and `Upper Bound`. Use `category_name` for every row's `Category`, and sort rows by `Average Score` from highest to lowest. Apply the same ordering to feature names, means, and both interval bounds so they stay aligned.

Return `(results_df, mean_scores, confidence_intervals)`. Keep `mean_scores` and `confidence_intervals` in the original `feature_names` order. Sort only `results_df` by `"Average Score"` for presentation. The row for a feature in `results_df` may therefore be different from its position in the returned arrays. For example, `confidence_intervals[:, j]` always describes `feature_names[j]`, regardless of where that feature appears in the sorted `results_df`.

An example output of `results_df` is below.

<div style="text-align: center;">
  <img src="https://i.imgur.com/obtOmGn.png" alt="Bootstrap Model Strengths" style="display: block; margin-left: auto; margin-right: auto; width: 75%;">
</div>

In [ ]:
N_BOOTSTRAP = 200

def get_bootstrapped_score(X, y, feature_names, category_name="Overall", n_bootstrap=N_BOOTSTRAP):
    """
    Bootstraps logistic regression model scores to estimate confidence intervals.
    Args:
        X: Feature matrix, shape (n_samples, n_features)
        y: Labels
        feature_names: Names for the columns of X, in column order — column j of X is
            the feature called feature_names[j]
        category_name: Value for the "Category" column, tagging which slice of the data
            this fit came from
        n_bootstrap: Number of bootstrap samples
    Returns:
        results_df: One row per feature, with columns Feature, Category, Average Score,
            Lower Bound, Upper Bound. Rows are sorted by Average Score, highest first.
        mean_scores: Mean bootstrapped coefficient per feature (np.array). Position j
            corresponds to the original column j of X, i.e. feature_names[j].
        confidence_intervals: 2.5 and 97.5 percentiles, np.array shape [2, n_features].
            Row 0 contains lower bounds; row 1 contains upper bounds.
            Column j corresponds to the original column j of X, same order as mean_scores.
    """
    # TODO
    np.random.seed(SEED)  # for reproducibility
    bootstrap_coefficients = []
    for i in range(n_bootstrap):
        indices = np.random.choice(len(X), size=len(X), replace=True)
        ...
        lr = ...
        lr.fit(...)
        ...

    # 3. Convert the collected coefficient vectors to an array. Rows are bootstrap
    #    replicates; columns follow the original feature_names order.
    bootstrap_coefficients = np.array(bootstrap_coefficients)  # shape: (n_bootstrap, n_features)
    # 4. Compute the mean bootstrapped strength for each feature; shape: (n_features,).
    mean_scores = ...
    # 5. Compute the 2.5th and 97.5th percentiles for each feature across
    #    replicates. Shape: (2, n_features); row 0 = lower bounds, row 1 = upper bounds.
    confidence_intervals = ...

    # 6. Build results_df from sorted copies, keeping the returned arrays in
    #    their original feature_names order.
    sorted_indices = ...  # Indices that sort mean_scores from highest to lowest.
    sorted_features = ...
    sorted_mean_scores = ...
    sorted_confidence_intervals = ...  # Reorder columns, keeping both bound rows.

    # Assemble one row per feature, using the sorted values above.
    results = {
        "Feature": ...,
        "Category": ...,          # category_name, once per row
        "Average Score": ...,
        "Lower Bound": ...,  # Row 0 of sorted_confidence_intervals
        "Upper Bound": ...,  # Row 1 of sorted_confidence_intervals
    }
    results_df = pd.DataFrame(results)
    return results_df, mean_scores, confidence_intervals

results_df, mean_scores, confidence_intervals = get_bootstrapped_score(
    X, y, selected_lmarena_models, n_bootstrap=N_BOOTSTRAP
)

# Rename the "Feature" column to "Model" because each feature in our input X was a +1/-1 representing
# which models were participated in a battle
results_df = results_df.rename(columns={"Feature": "Model"})

# Test that confidence intervals make sense
assert (confidence_intervals[0] <= confidence_intervals[1]).all(), "Every lower bound must be <= upper bound."
assert ((confidence_intervals[0] <= mean_scores) & (mean_scores <= confidence_intervals[1])).all(), "Each mean score should lie within its CI."

In [ ]:
grader.check("q5a")

Now let's visualize the intervals! *🧙*

In [ ]:
# Run this cell, no code changes required :)

bootstrap_scores_fig = go.Figure()
# Use the sorted values from results_df for plotting
bootstrap_scores_fig.add_trace(go.Scatter(
    x=results_df["Model"],
    y=results_df["Average Score"],
    mode='markers',
    name='Model Scores',
    marker={'size': 5, 'color': 'blue'},
    error_y={'type': 'data',
            'array': results_df["Upper Bound"] - results_df["Average Score"],   # Upper error
            'arrayminus': results_df["Average Score"] - results_df["Lower Bound"],  # Lower error
            'visible':True,
            }
    )
)

bootstrap_scores_fig.update_layout(
    title='Model Performance Scores with 95% Confidence Intervals (Sorted by Mean Score)',
    xaxis_title='Models',
    yaxis_title='Score',
    xaxis={'tickangle': 45},
    height=500
)

if not IS_GRADING_ENV:
    bootstrap_scores_fig.show()

## Question 5b: Confidence-Aware Ranks

Now that we have confidence intervals, we can assign a rank to each model. We want the rank of model $i$ to represent the number of models that are **confidently better** than model $i$.

When we say model A is **confidently better** than model B, it will mean that model A's lower bound is still greater than model B's upper bound. Remember that greater rank means that there are more models that perform better than the current model.

Implement the `assign_rank` function below that assigns rank to the model.

In [ ]:
def assign_rank(row, df=results_df):
    """
    Input:
        row : pd.Series
            A row of the DataFrame (representing a model’s metrics).
        df : pd.DataFrame (default = results_df)
            DataFrame containing model performance with 'Lower Bound' and 'Upper Bound'.

    Output:
        int : The rank of the model, defined as (# of models confidently better) + 1.
    """

    count = ...
    return ...


results_df['Rank'] = results_df.apply(lambda r: assign_rank(r, results_df), axis=1)
results_df = results_df.sort_values(by="Rank", ascending=True)
results_df

In [ ]:
grader.check("q5b")

Let's visualize the results of the new ranks! We've provided a helper called `plot_rank_heatmap` in the following cell. All you have to do is run the cell :)

In [ ]:
# Run this cell, no code changes required :)

def plot_rank_heatmap(df: pd.DataFrame, title: str = "Rank Heatmap", top_n: int = 20, 
                     categories: list[str] | None = None, selected_models: list[str] | None = None) -> go.Figure:
    """
    Staff-provided helper
    Create a heatmap showing ranks across categories using tidy data format.
    
    Args:
        df: pd.DataFrame
            Tidy DataFrame with columns ['Model', 'Rank', 'Category']
        title: str
            Title of the heatmap
        top_n: int
            Number of top models to show (default 20)
        categories: list[str], optional
            List of categories to include. If None, uses all categories in df
    
    Returns:
        plotly.graph_objects.Figure
            Interactive heatmap showing model ranks across categories
    """
    # Validate input format
    required_cols = ['Model', 'Rank', 'Category']
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")
    
    # Filter categories if specified
    if categories is not None:
        df = df[df['Category'].isin(categories)].copy()
        
    if selected_models is not None:
        df = df[df['Model'].isin(selected_models)].copy()
    
    # Get the categories in the data (sorted for consistency)
    all_categories = sorted(df['Category'].unique())
    
    # Find top models based on overall ranking (or first category if no "Overall")
    if 'Overall' in df['Category'].values:
        top_models_df = df[df['Category'] == 'Overall'].nsmallest(top_n, 'Rank')
    else:
        # Use the first category alphabetically
        first_category = all_categories[0]
        top_models_df = df[df['Category'] == first_category].nsmallest(top_n, 'Rank')
    
    top_models = top_models_df['Model'].tolist()
    
    # Filter to top models and pivot to wide format for heatmap
    filtered_df = df[df['Model'].isin(top_models)].copy()
    
    # Pivot tidy data to wide format for heatmap
    pivot_df = filtered_df.pivot(index='Model', columns='Category', values='Rank')
    
    # Reorder models by their overall rank (or first category rank)
    if 'Overall' in pivot_df.columns:
        model_order = pivot_df.sort_values('Overall')['Overall'].index.tolist()
    else:
        first_category = all_categories[0]
        model_order = pivot_df.sort_values(first_category)[first_category].index.tolist()
    
    # Reorder rows and columns
    pivot_df = pivot_df.loc[model_order, all_categories]
    
    # Prepare data for heatmap
    rank_data = pivot_df.values
    models = pivot_df.index.tolist()
    category_names = pivot_df.columns.tolist()
    
    # Reverse model order so best models appear at top
    models.reverse()
    rank_data = rank_data[::-1]
    
    fig = go.Figure(data=go.Heatmap(
        z=rank_data,
        x=category_names,
        y=models,
        colorscale='Viridis_r',  # darker = better rank
        text=rank_data,
        texttemplate="%{text}",
        textfont={"size": 10},
        hovertemplate='Model: %{y}<br>Category: %{x}<br>Rank: %{z}<extra></extra>',
        showscale=False
    ))

    fig.update_layout(
        title=f'Rank Heatmap for Top {top_n} Models {title}',
        xaxis_title='Category',
        yaxis_title='Model',
        height=max(400, len(models) * 32) 
    )
    
    return fig

fig = plot_rank_heatmap(results_df)
fig.show()

# Question 6: Style-Controlled Model Rankings

The leaderboard above estimates a strength for each model from wins and losses. However, users may also respond to presentation choices such as response length, headings, lists, or bold text. for example, one thing that has been known to affect user preference is response length: people (and LLMs) tend to prefer longer answers. A recurring observation in human grading and UX is that **longer responses are often preferred**. Analyses from the SAT essay reported that **essay length strongly correlated with higher scores—even when errors were present** ([New York Times, 2005](https://www.nytimes.com/2005/05/04/education/sat-essay-test-rewards-length-and-ignores-errors.html)). 

If these presentation choices affect votes, part of a model's estimated strength may reflect style rather than the underlying quality we hoped to measure.

In the remainder of this homework, we will analyze how style features affect a model's strength! We will include response-style features, such as length and formatting, in the logistic-regression model. This lets us estimate model strength while accounting for the measured differences in style between responses.

A caveat to keep in mind is that because the data are observational, the results show associations rather than proving that a particular style causes a model to win (recall age-old warings about correlation vs. causation!). However, there is still a lot we can discover about LLM behavior and human preferences through this investigation :)

## Question 6a: Extract style metrics

Each battle contains precomputed response statistics in the `conv_metadata` column. Build a dataframe containing the raw metrics for model A and model B of each row: bold text count, header count, list count, and assistant-token count.

Implement `extract_style_metrics` so that it returns one row per battle and separate columns for the A and B values.

The result of `extract_style_metrics` should have 8 columns:
* `bold_count_a`
* `bold_count_b`
* `header_count_a`
* `header_count_b`
* `list_count_a`
* `list_count_b`
* `sum_assistant_a_tokens`
* `sum_assistant_b_tokens`

The function should not mutate its input DataFrame.

**Hint:**
* `bold_count_X`, `header_count_X`, and `list_count_X` are dictionaries with multiple keys and values. To compute the metrics for these dictionary-typed metadata fields, sum all the values of each dictionary.

In [ ]:
STYLE_METRIC_COLUMNS = ['bold_count_a', 'bold_count_b',
                        'header_count_a', 'header_count_b',
                        'list_count_a', 'list_count_b',
                        'sum_assistant_a_tokens', 'sum_assistant_b_tokens']

def extract_style_metrics(df):
    """
    For each battle row, extract the conv metadata metrics for A and B separately.
    """
    ...

style_metrics = extract_style_metrics(selected_battles_no_ties)
style_metrics.head()

In [ ]:
grader.check("q6a")

## Question 6b: Calculate normalized style differences

Now, we want to normalize the differences to see how much side A differs from side B in a given battle for each style metric. We will use the normalized difference formula:

$$
\operatorname{normdiff}(a,b)=
\begin{cases}
0 & a+b=0, \\
\dfrac{a-b}{a+b} & \text{otherwise}.
\end{cases}
$$


The intuition behind the normalized difference formula is that it is only dependent on the ratio between $a$ and $b$ rather than their magnitudes.

Consider bold-heading counts across three battles:

<div align="center">

| Battle | A | B | raw `a - b` | `normdiff(a, b)` |
|---|---|---|---|---|
| 1 | 5 | 6 | −1 | −0.091 |
| 2 | 50 | 60 | −10 | −0.091 |
| 3 | 500 | 501 | −1 | −0.001 |

</div>

B uses bold headings more often than A by the same *proportion* in battles 1 and 2 (the ratio of B:A is 6:1 in both battles 1 and 2). But if we looked only at raw difference counts, in B used 10x more bold headings in battle 2 than battle 1. Battles 1 and 3 share a raw difference of −1, though the difference of just one heading is much more significant when there are only 5 or 6 headings total compared to 500 or 501 headings. Thus, the normalized difference gets both
cases right because it depends only on the ratio of the counts:

$$\operatorname{normdiff}(a,b) = \frac{a-b}{a+b} = \frac{a/b - 1}{a/b + 1}$$

This buys us three things:

- **A shared scale:** Values fall in $[-1, +1]$, so token counts in the thousands and
  heading counts in single digits yield comparable coefficients, and no single lopsided
  battle dominates the fit.
- **Comparability:** We only care about how much A and B differ from each other in the same battle. The normalized difference lets us compare the ratios of A's and B's style uses across battles.
- **Antisymmetry:** $\operatorname{normdiff}(b, a) = -\operatorname{normdiff}(a, b)$, so the
  B-to-A row is the exact negation of the A-to-B row, matching the $\pm 1$ encoding from
  Question 4.

Use the `style_metrics` dataframe you built in question 6a.
1. Implement `normalized_style_difference`.
2. Then, implement `make_style_feature_matrix`, which should create a NumPy array that stores the normalized difference between A's and B's style metrics. You should use your implementation of `normalized_style_difference` as a helper! Each row should contain four features, in the order `style_bold_count`, `style_header_count`, `style_list_count`, and `style_sum_assistant_tokens`, as defined by `STYLE_FEATURE_NAMES`.

For instance, the entry corresponding to `style_bold_count` should contain:
$$
\text{normdiff}(style\_bold\_count\_a, style\_bold\_count\_b) =
\begin{cases}
0 & \text{if } style\_bold\_count\_a + style\_bold\_count\_b = 0 \\[6pt]
\dfrac{style\_bold\_count\_a - style\_bold\_count\_b}{style\_bold\_count\_a + style\_bold\_count\_b} & \text{otherwise}
\end{cases}
$$

Just like in question 4 how we derived two feature vectors from each battle, each battle should have two corresponding rows in the style feature matrix: an A-to-B row and a B-to-A row. The B-to-A row's normalized style difference values should be the negation of the A-to-B row's.

In [ ]:
# List of triplets for reference! These might come in handy for your implementations:
# - the first elem in each triplet is the col name of the normalized style difference
# - the 2nd and 3rd elem are the col names of A's metric and B's metric
STYLE_FEATURE_TRIPLETS = [
    ("style_bold_count", "bold_count_a", "bold_count_b"),
    ("style_header_count", "header_count_a", "header_count_b"),
    ("style_list_count", "list_count_a", "list_count_b"),
    ("style_sum_assistant_tokens", "sum_assistant_a_tokens", "sum_assistant_b_tokens"),
]
STYLE_FEATURE_NAMES = [name for name, _, _ in STYLE_FEATURE_TRIPLETS]

def normalized_style_difference(a, b):
    """Return the normalized A-minus-B difference."""
    ...

def make_style_feature_matrix(df, style_metrics=None):
    """Return style rows aligned with the doubled rows produced in Question 4."""
    ...

X_style = make_style_feature_matrix(selected_battles_no_ties, style_metrics)
print("Model features:", X.shape)
print("Style features:", X_style.shape)


In [ ]:
grader.check("q6b")

In [ ]:
def plot_rank_changes(comparison, title):
    """Plot baseline and controlled ranks for the models with the largest changes."""
    plot_df = comparison.copy()
    plot_df["Rank Change"] = plot_df["Controlled Rank"] - plot_df["Baseline Rank"]
    plot_df["Absolute Rank Change"] = plot_df["Rank Change"].abs()
    plot_df = plot_df.sort_values("Absolute Rank Change", ascending=False).head(10)
    plot_df = plot_df.sort_values("Controlled Rank", ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    y_pos = np.arange(len(plot_df))
    ax.hlines(y_pos, plot_df["Baseline Rank"], plot_df["Controlled Rank"], color="0.75")
    ax.scatter(plot_df["Baseline Rank"], y_pos, label="Baseline", s=45)
    ax.scatter(plot_df["Controlled Rank"], y_pos, label="Style controlled", s=45)
    ax.set_yticks(y_pos, plot_df["Model"])
    ax.set_xlabel("Rank (smaller is better)")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    return fig


## Question 6c: Fit a style-controlled leaderboard

In Question 4, `X` recorded which LLMs participated in each battle. Each row represents one orientation of a battle: the LLM on side A has a `+1`, the LLM on side B has a `-1`, and the other LLM columns have `0`. The two orientations give two rows per battle.

In Question 6b, `X_style` recorded the stylistic differences between the two responses, using the same row ordering. For example, a positive `style_sum_assistant_tokens` value means that side A's response is longer than side B's response.

We can combine both the LLM-identity information (i.e. which models partook in a battle) with the stylistic differences we calculated earlier for each battle by concatenating the matrices column-wise:

```python
X_with_style = np.concatenate([X, X_style], axis=1)
```

In the resulting, combined matrix, each row corresponds to a single battle. This row has columns denoting which LLMs were involved in this battle, as well as style columns that contain the normalized style features comparing how often a certain stylistic effect occurred in side A's response compared to side B's response in this battle.

The combined matrix lets logistic regression estimate each LLM's strength while accounting for the measured style differences. Use `get_bootstrapped_score` with `feature_names = selected_models + STYLE_FEATURE_NAMES` to fit this model repeatedly on bootstrap samples. The bootstrap results include one coefficient for every LLM and every style feature.

Use the `Feature` column to separate the results into two groups:

- LLM features: labels in `selected_models`. Their coefficients are the style-adjusted model strength estimates, so use these rows to assign leaderboard ranks.
- Style features: labels in `STYLE_FEATURE_NAMES`. Their coefficients describe the association between each measured style difference and the probability that side A wins. Keep these rows separate and do not rank them as LLMs.

Finally, compare the controlled LLM ranks with the baseline ranks from Question 5. You may rename the LLM-only `Feature` column to `Model` when preparing the rank-comparison plot. Run the completed cell to save `written/q6c.png`.

In [ ]:
def fit_style_controlled_leaderboard(
    X_model, y, X_style, selected_models, style_feature_names, n_bootstrap=N_BOOTSTRAP
):
    X_with_style = np.concatenate([X_model, X_style], axis=1)
    # TODO: call get_bootstrapped_score and pass in both selected_models and the style_feature_names
    ...

    # TODO: separate out the rows from combined_results where the "Feature" column is in selected_models
    model_results = ...

    # TODO: Use assign_rank to rank the models. Then sort the models by their rank in ascending order
    ...
    
    # TODO: separate out the rows from combined_results where the "Feature" column is a style_feature_name
    style_results = ...
    return model_results, style_results


style_model_results, style_coefficient_results = fit_style_controlled_leaderboard(
    X, y, X_style, selected_lmarena_models, STYLE_FEATURE_NAMES
)
style_rank_comparison = (
    results_df[["Model", "Rank"]].rename(columns={"Rank": "Baseline Rank"})
    .merge(
        style_model_results[["Feature", "Rank"]].rename(
            columns={"Feature": "Model", "Rank": "Controlled Rank"}
        ),
        on="Model", how="inner"
    )
)
display(style_coefficient_results)
display(style_rank_comparison.sort_values("Controlled Rank"))

fig_q6c = plot_rank_changes(
    style_rank_comparison,
    "Largest rank changes after controlling for response style",
)
save_figure(fig_q6c, "q6c")
fig_q6c.show()


In [ ]:
grader.check("q6c")

# Question 7: Build Your Own Style Feature

The supplied metadata captures only a few visible formatting choices, such as when the LLM uses bold text markers. But there's lots of other stylistic choices that users might explicitly or implicitly prefer! Now it's your turn to identify a style feature and measure its effect on LLM strength. You can try reading through some LLM responses on your own, playing with Arena AI's head-to-head chats yourself, or your past experience to propose an interpretable style feature.

## Question 7a: Define and Measure a Custom Style Feature

1. Pick a stylistic feature you want to analyze: for isnstance, presence of certain phrases, punctuation, formatting, etc. You can check multiple different phrases if you want, they just to have a common "theme" (e.g. phrases that all contain GenZ slang).
2. Set `CUSTOM_STYLE_NAME` to a short, descriptive name for your feature.
3. Implement `custom_style_value(conversation)`. It should inspect only the assistant's messages and return a finite, nonnegative number measuring how strongly the feature appears in that conversation. Your feature must be nonconstant across the selected battles. It should vary from battle to battle, rather than being the same number for all battles — otherwise, it wouldn't be a very helpful signal to tell one battle/model-pairing apart from another!
    - We've provided `assistant_text`, which joins all assistant messages in one conversation into a single string. You can use this helper function when implementing `custom_style_value`.
4. For each battle, apply `custom_style_value` separately to `conversation_a` and `conversation_b`. Call the results `value_a` and `value_b`.
5. Compute `normalized_style_difference(value_a, value_b)`. Store this A-versus-B difference in a new column of `selected_battles_no_ties` whose name is `CUSTOM_STYLE_NAME`.
6. Build the rows of `X_custom_style` using the same two-orientation arrangement as the style features in Question 6b. For each battle, append the A-versus-B difference first, followed immediately by its negation for the B-versus-A orientation.
7. Convert the collected rows to a NumPy array with shape `(2 * number_of_battles, 1)`.

The provided code after your implementation will append `X_custom_style` to `X_style`, refit the style-controlled leaderboard, and save `written/q7a.png`.

This is open ended, so we encourage you to get creative with it!

**Important grading notes:**
- You cannot use a feature already mentioned explicitly in the problem preamble or explored (e.g. response length, bold text, headings, lists).

In [ ]:
def assistant_text(conversation):
    """
    Staff-provider helper!
    Join only assistant messages from an Arena conversation.
    """
    return "\n\n".join(
        message.get("content", "")
        for message in conversation
        if message.get("role") == "assistant"
    )


# TODO: Give your custom style feature a short, descriptive name.
CUSTOM_STYLE_NAME = "your_feature_name"

def custom_style_value(conversation):
    # TODO: Measure your chosen style in one conversation.
    # Return a finite, nonnegative number and inspect assistant messages only.
    ...

# TODO: Create the DataFrame column that will hold one normalized
# A-versus-B difference per battle.
selected_battles_no_ties[CUSTOM_STYLE_NAME] = ...

# TODO: Initialize a list for the two oriented rows from every battle.
custom_style_rows = ...

# TODO: Iterate through the battles in their existing DataFrame order.
for battle_index, battle in ...:
    # TODO: Evaluate your feature on side A's conversation.
    value_a = ...
    # TODO: Evaluate your feature on side B's conversation.
    value_b = ...
    # TODO: Compute the normalized A-versus-B difference.
    difference = ...
    # TODO: Store the difference in this battle's DataFrame row.
    ...
    # TODO: Append A-versus-B first and B-versus-A second.
    ...

# TODO: Convert the rows to a float array with one column.
X_custom_style = ...

X_style_with_custom = np.column_stack([X_style, X_custom_style])
STYLE_FEATURE_NAMES_WITH_CUSTOM = STYLE_FEATURE_NAMES + [CUSTOM_STYLE_NAME]

custom_model_results, custom_style_coefficient_results = (
    fit_style_controlled_leaderboard(
        X, y, X_style_with_custom,
        selected_lmarena_models, STYLE_FEATURE_NAMES_WITH_CUSTOM
    )
)
custom_rank_comparison = (
    results_df[["Model", "Rank"]].rename(columns={"Rank": "Baseline Rank"})
    .merge(
        custom_model_results[["Feature", "Rank"]].rename(
            columns={"Feature": "Model", "Rank": "Controlled Rank"}
        ),
        on="Model", how="inner"
    )
)
fig_q7a = plot_rank_changes(
    custom_rank_comparison,
    f"Rank changes with custom style feature: {CUSTOM_STYLE_NAME}",
)
if not IS_GRADING_ENV:
    fig_q7a.show()
save_figure(fig_q7a, "q7a")

In [ ]:
grader.check("q7a")

## Question 7b: Reflect on your feature

Inspect the custom-feature coefficient, its confidence interval, and the rank comparison. Explain:

1. why you chose the feature and how your implementation measures it;
2. whether its logistic regression coefficient (positive or negative) supports your hypothesis;
3. which models changed most after controlling for it;
4. one plausible confounder, measurement limitation, or failure mode.

Run the two inspection cells below before answering. If the custom feature's confidence interval includes zero, do not treat its coefficient as strong evidence of an association. Note that the controlled ranks account for all style features in the model, including your custom feature.

Write your answer in `q7b_answer` and run the cell to save it for the written submission.

### Inspect your results

The first table shows your custom feature's estimated coefficient and 95% bootstrap confidence interval. A positive coefficient means that a larger A-versus-B value for your feature is associated with a higher probability that side A wins; a negative coefficient means the opposite.

In [ ]:
# Run this cell, no code changes required :)
custom_feature_summary = custom_style_coefficient_results.loc[
    custom_style_coefficient_results["Feature"] == CUSTOM_STYLE_NAME,
    ["Feature", "Average Score", "Lower Bound", "Upper Bound"],
].reset_index(drop=True)
display(custom_feature_summary)

The next table compares each model's baseline rank with its rank after controlling for all measured style features, including your custom feature. A positive rank change means that the model moved down the leaderboard; a negative rank change means that it moved up.

In [ ]:
# Run this cell, no code changes required :)
custom_rank_changes = custom_rank_comparison.copy()
custom_rank_changes["Rank Change"] = (
    custom_rank_changes["Controlled Rank"]
    - custom_rank_changes["Baseline Rank"]
)
custom_rank_changes["Absolute Rank Change"] = (
    custom_rank_changes["Rank Change"].abs()
)
display(
    custom_rank_changes.sort_values(
        ["Absolute Rank Change", "Controlled Rank"],
        ascending=[False, True],
    ).reset_index(drop=True)
)

<!-- BEGIN QUESTION -->



In [ ]:
# TODO: Write your answer in q7b_answer, then run this cell to save it.
q7b_answer = """
1. YOUR ANSWER HERE

2. YOUR ANSWER HERE

3. YOUR ANSWER HERE

4. YOUR ANSWER HERE
"""
save_written_answer(q7b_answer, "q7b")


<!-- END QUESTION -->

# Before you submit

Save the notebook, run every cell that produces a written answer or figure, and then run the check below. Missing files under `written/` print a warning but do not fail the cell.

In [ ]:
warn_missing_written_files()

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

The cell below will generate a zip file for you to submit. **If you are working in Google Colab, make sure to download the generated zip file from your Google Drive folder where your Colab notebook is located! The link to download directly from the notebook won't work in Google Colab.**
It will also include all completed responses to manually graded questions saved under `written/` and other required files. **Please save before exporting!**

Then, make sure to submit the **generated zip file** to the coding assignment on Pensive.
Pensive is set up to automatically format the files inside the zip file's `written/` directory into a PDF and submit it to the corresponding coding PDF assignment autoomatically.
After the autograder finishes running, please double-check the submission that it automatically generates for the coding PDF assignment.

Note: Both this final submission generation cell and the autograder expect the assignment notebook to have the same name as the input to `otter.Notebook(...)` in the Otter initialization cell at the top of the notebook. Do not rename your notebook, otherwise the final export and the autograder may run into errors processing your submission.


In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True, files=['written'])